In [1]:
from utils.utils import load_from_pickle
import pandas as pd

# pd.set_option('display.max_colwidth', None)

testset = load_from_pickle('filtered_testset')
dataset = testset

In [2]:
dataset

,question,contexts,ground_truth,evolution_type,metadata,episode_done,cleaned_question
0,"Here is the rewritten question:\n\n""What indiv...",[DOC NAME: Undang-undang Nomor 42 Tahun 2008 T...,Pemantau asing yang berasal dari perwakilan ne...,simple,[{'title': 'Undang-undang Nomor 42 Tahun 2008 ...,True,What individuals or entities have the right to...
1,"Here is the rewritten question:\n\n""What does ...",[DOC NAME: Undang-undang Nomor 42 Tahun 2008 T...,The term 'perolehan suara yang lebih luas seca...,simple,[{'title': 'Undang-undang Nomor 42 Tahun 2008 ...,True,What does the term 'perolehan suara yang lebih...
2,"Here is the rewritten question:\n\n""What facto...","[DOC NAME: | \n\n2009, No.9 \n43 \n \n \n \n ...",The factors that influenced the performance of...,simple,"[{'title': '', 'jenis_bentuk_peraturan': '', '...",True,What factors influenced the performance of the...
3,"Here is the rewritten question:\n\n""What is th...","[DOC NAME: | \n\n2009, No.9 \n261 \nwww.perat...",The Ikhtisar Laporan Keuangan Perusahaan Negar...,simple,"[{'title': '', 'jenis_bentuk_peraturan': '', '...",True,What is the purpose of the Ikhtisar Laporan Ke...
4,Here is a question that can be fully answered ...,[DOC NAME: | \n\nPRESIDEN\nREPUBLIK INDONESIA...,The outcome of the Conference for the Adoption...,simple,"[{'title': '', 'jenis_bentuk_peraturan': '', '...",True,What was the outcome of the Conference for the...
...,...,...,...,...,...,...,...
2639,Here is a question that can be fully answered ...,[DOC NAME: Undang-undang Nomor 20 Tahun 1953 T...,The purpose of the pengesahan of the Perjanjia...,simple,[{'title': 'Undang-undang Nomor 20 Tahun 1953 ...,True,According to the Undang-Undang Nomor 20 Tahun ...
2640,Here is a question that can be fully answered ...,[DOC NAME: Undang-undang Nomor 19 Tahun 1997 T...,The reason for revising the law on tax collect...,simple,[{'title': 'Undang-undang Nomor 19 Tahun 1997 ...,True,"According to the context, what is the reason f..."
2641,"Here is the rewritten question:\n\n""What are t...",[DOC NAME: Undang-undang Nomor 56 Tahun 2024 T...,The subdistricts that make up the city of Paya...,simple,[{'title': 'Undang-undang Nomor 56 Tahun 2024 ...,True,What are the subdistricts that make up the cit...
2642,Here is a question that can be fully answered ...,[DOC NAME: Undang-undang Nomor 56 Tahun 2024 T...,The 5 Kecamatan that make up the city of Payak...,simple,[{'title': 'Undang-undang Nomor 56 Tahun 2024 ...,True,According to the Undang-undang Nomor 56 Tahun ...


In [3]:
for i, (q, a) in enumerate(zip(dataset['cleaned_question'], dataset['ground_truth'])):
    print(f"Question {i+1}: {q}\n")
    print(f"Answer: {a}")
    print("-" * 40)

Question 1: What individuals or entities have the right to access information from the KPU, KPU provinsi, and KPU kabupaten/kota, according to the Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden?

Answer: Pemantau asing yang berasal dari perwakilan negara asing yang berstatus diplomat berhak atas kekebalan diplomatik selama menjalankan tugas sebagai pemantau Pemilu Presiden dan Wakil Presiden, dan pemantau Pemilu Presiden dan Wakil Presiden yang mematuhi peraturan perundang-undangan dan menghormati kedaulatan Negara Kesatuan Republik Indonesia.
----------------------------------------
Question 2: What does the term 'perolehan suara yang lebih luas secara berjenjang' mean in the context of the presidential election, as defined in the Indonesian law on presidential and vice-presidential elections?

Answer: The term 'perolehan suara yang lebih luas secara berjenjang' refers to the candidate who wins in more provinces, as defined in the Indonesian law o

In [4]:
dataset2 = dataset[:5]
questions = list(dataset2["cleaned_question"])

In [5]:
dataset2.head(1)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,cleaned_question
0,"Here is the rewritten question:\n\n""What indiv...",[DOC NAME: Undang-undang Nomor 42 Tahun 2008 T...,Pemantau asing yang berasal dari perwakilan ne...,simple,[{'title': 'Undang-undang Nomor 42 Tahun 2008 ...,True,What individuals or entities have the right to...


In [6]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config
from utils.chain.rag import create_combined_answer_chain


llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu-all").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

/raid/p-storage/joshuamihai_e64ef832/.local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loaded index 'uu-all'.


In [7]:
# result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})
# result

In [8]:
from langchain_core.prompt_values import StringPromptValue
from langchain_core.documents.base import Document

def unique_context(context: list):
    context_unique = []
    
    for ctx in context:
        # Handle cases where context is a list of multiple documents/values
        if isinstance(ctx, list):
            unique_items = []
            for item in ctx:
                if isinstance(item, StringPromptValue):
                    content = item.text
                elif isinstance(item, Document):
                    content = item.page_content
                else:
                    continue
                
                # Add unique content only
                if content not in unique_items:
                    unique_items.append(content)
            context_unique.append(unique_items)
        else:
            # If it's a single context, process as a single string or document
            if isinstance(ctx, StringPromptValue):
                context_unique.append(ctx.text)
            elif isinstance(ctx, Document):
                context_unique.append(ctx.page_content)
            else:
                context_unique.append(ctx)  # In case there are other types

    return context_unique

# Chain

In [9]:
answer = []
context = []

In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
i = 1

for question in questions:
    try:
        result = chain_quality.invoke({"question":question, "chat_history":"-"})
        print("Successfully run the chain for question", i)
        answer.append(result['answer'])
        context.append(result['context'])
        i += 1
    
    except Exception:
        print("Failed run the chain for question", i)
        break



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '42', tahun: '2008', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:MENGUBAH|DIUBAH|MENCABUT|DICABUT]->(r:Peraturan)
WHERE r.tentang CONTAINS 'KPU' OR r.tentang CONTAINS 'KPU provinsi' OR r.tentang CONTAINS 'KPU kabupaten/kota'
RETURN DISTINCT r.pemrakarsa AS pemrakarsa

Full Context:
[]

> Finished chain.
Successfully run the chain for question 1


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {jenis_peraturan: 'UNDANG-UNDANG', tentang: 'PILPRES'})
MATCH (p)-[:MENGUBAH|DIUBAH|MENCABUT|DICABUT]->(r:Peraturan)
WHERE r.tentang = 'perolehan suara yang lebih luas secara berjenjang'
RETURN 
  p AS regulation, 
  r AS relatedRegulation

Full Context:
[]

> Finished chain.
Successfully run the chain for question 2


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {tahun: '2006'})
MATCH (p)-[:MENGUBAH|DIUBAH|MENCABUT|DI

In [12]:
len(context)

5

In [13]:
context_unique = unique_context(context)
# context_unique[1]

In [14]:
ground_truth = list(dataset2["ground_truth"])
# ground_truth[1]

In [15]:
answer[0]

'Based on the provided context, the Law Number 42 of 2008 Regarding Presidential and Vice Presidential Elections does not explicitly mention the individuals or entities that have the right to access information from the KPU, KPU provinsi, and KPU kabupaten/kota. However, we can infer from the documents that the following individuals or entities have the right to access information:\n\n* Saksi Pasangan Calon tingkat provinsi\n* Saksi Pasangan Calon tingkat kabupaten/kota\n* Panwaslu provinsi\n* Panwaslu kabupaten/kota\n* Bawaslu\n\nThis information can be found in the documents:\n\n* Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden (page 38)\n* Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden (page 40)\n* Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden (page 43)\n\n**Sources**:\n\n- Undang-undang Nomor 42 Tahun 2008 Tentang Pemilihan Umum Presiden dan Wakil Presiden (https://pera

In [16]:
# context_unique_str = []

# for i in context_unique:
#     ls = []
#     for j in i:
#         ls.append(j.page_content)
#     context_unique_str.append(ls)
    
context_unique_str = ['\n'.join(ctx) if isinstance(ctx, list) else str(ctx) for ctx in context_unique]
context_as_list = [[ctx] if isinstance(ctx, str) else ctx for ctx in context_unique]

In [17]:
len(context_unique)

5

In [18]:
len(context_unique_str[0])

14625

In [19]:
from datasets import Dataset

dict_eval = {
    "question": questions,
    "answer": answer,
    "contexts": context_as_list,
    "ground_truth": ground_truth
}

dataset = Dataset.from_dict(dict_eval)

In [20]:
import nest_asyncio
import warnings

nest_asyncio.apply()
warnings.filterwarnings("ignore")

In [21]:
from ragas.metrics import (FaithulnesswithHHEM, answer_relevancy, context_precision, context_recall, context_utilization, context_entity_recall,
noise_sensitivity_relevant, noise_sensitivity_irrelevant, answer_similarity, answer_correctness, faithfulness)
from ragas import evaluate

# faithfulness_with_hhem = FaithulnesswithHHEM()

metrics = [answer_relevancy, context_precision, context_recall, context_utilization, context_entity_recall,
           answer_similarity, answer_correctness, faithfulness]

metrics = [context_recall]

In [22]:
score = evaluate(dataset, metrics, llm, embed_mode, l)

NameError: name 'embed_mode' is not defined

In [ ]:
score

In [ ]:
# using 30 qs
score

In [ ]:
# using 5 qs
# score